<a href="https://colab.research.google.com/github/prasa129/Fun/blob/main/Monty_Hall_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Bayesian Solution to Monty Hall Problem

Ram Prasad

09-08-2025

Setup for Monty Hall problem:

- 3 closed doors (A, B, C). Behind one is a car, the other two, goats.

- You pick a door (e.g. A).

- The host opens one of the remaining doors (e.g. B, C) to reveal a goat, but will not open the door concealing the car.

- You can switch or keep your original choice.

Intuitively, switching wins with $p=2/3$, keeping your original choice wins with $p=1/3$. If you picked correctly (with probability 1/3) and switch, you lose. If you picked incorrectly (with probability 2/3) and switch, you win.

I provide a formal solution with an accompanying simulation.

Formal Solution:

Define three events, with corresponding priors:

- $C_{a}$: car is behind door A, $P(C_{a}) = \pi_{a}$

- $C_{b}$: car is behind door B, $P(C_{b}) = \pi_{b}$

- $C_{c}$: car is behind door C, $P(C_{c}) = \pi_{c}$

such that $\pi_{a} + \pi_{b} + \pi_{c}=1$. Assume your initial pick is $A$. The argument follows symmetrically for other picks. There are three cases:

- The car is behind A. Monty can open C with probability $\alpha$ and B with probability $1-\alpha$.

- The car is behind B. Monty opens C.

- The car is behind C. Monty opens B.

Compute posteriors, conditioning on Monty opening door C. Call this event M = C. By the law of total probability,

\begin{align*}
P(M=C) &= P(M=C|C_{a})P(C_{a}) + P(M=C|C_{b})P(C_{b}) + P(M=C|C_{c})P(C_{c}) \\
       &= \alpha \pi_{a} + 1 \cdot \pi_{b} + 0 \cdot \pi_{c} \\
       &= \alpha \pi_{a} + \pi_{b} \\
\end{align*}

and Bayes law,

\begin{align*}
P(C_{a} | M = C) &= \frac{P(M=C|C_{a})P(C_{a})}{P(M=C)} \\
&= \frac{\alpha \pi_{a}}{\alpha \pi_{a} + \pi_{b}} \\ \\
P(C_{b} | M = C) &= \frac{P(M=C|C_{b})P(C_{b})}{P(M=C)} \\
&= \frac{ \pi_{b}}{\alpha \pi_{a} + \pi_{b}} \\
\end{align*}

Compute posteriors, conditioning on Monty opening door B. Call this event M = B. By the law of total probability:

\begin{align*}
P(M=B) &= P(M=B|C_{a})P(C_{a}) + P(M=B|C_{b})P(C_{b}) + P(M=B|C_{c})P(C_{c}) \\
       &= (1-\alpha) \pi_{a} + 0 \cdot \pi_{b} + 1 \cdot \pi_{c} \\
       &= (1-\alpha) \pi_{a} + \pi_{c} \\
\end{align*}

and Bayes law:

\begin{align*}
P(C_{a} | M = B) &= \frac{P(M=B|C_{a})P(C_{a})}{P(M=B)} \\
&= \frac{(1-\alpha) \pi_{a}}{(1-\alpha) \pi_{a} + \pi_{c}} \\
P(C_{c} | M = B) &= \frac{P(M=B|C_{c})P(C_{c})}{P(M=C)} \\
&= \frac{ \pi_{c}}{(1-\alpha) \pi_{a} + \pi_{c}} \\
\end{align*}

In the classic Monty Hall problem, $\alpha=0.5$ and $\pi_{a}=\pi_{b}=\pi_{c}=1/3$, thus $\pi_{b} > \alpha \pi_{a}$ and $\pi_{c} > (1-\alpha)\pi_{a}$. Switching is optimal, regardless of which door Monty reveals:

\begin{align*}
P(C_{a} | M = C) &= \frac{\frac{1}{2} \cdot \frac{1}{3}}{\frac{1}{2} \cdot \frac{1}{3} + \frac{1}{3}} = \frac{1}{3} \\
P(C_{b} | M = C) &= \frac{\frac{1}{3}}{\frac{1}{2} \cdot \frac{1}{3} + \frac{1}{3}} = \frac{2}{3}\\
P(C_{a} | M = B) &= \frac{(1-\frac{1}{2}) \cdot \frac{1}{3}}{(1-\frac{1}{2}) \cdot \frac{1}{3} + \frac{1}{3}} = \frac{1}{3} \\
P(C_{c} | M = B) &= \frac{\frac{1}{3}}{(1-\frac{1}{2}) \frac{1}{3} + \frac{1}{3}} = \frac{2}{3} \\
\end{align*}

When the posterior probabilities are equal, you are indifferent between switching or keeping your original choice. Indifference is a function of the priors and bias probability $\alpha$. When Monty reveals C:

\begin{align*}
P(C_{a} | M = C) &= P(C_{b}|M=C) \\
\frac{\alpha \pi_{a}}{\alpha \pi_{a} + \pi_{b}} &= \frac{\pi_{b}}{\alpha \pi_{a} + \pi_{b}} \\
\pi_{b} &= \alpha \pi_{a} \\
\end{align*}

and when Monty reveals B:

\begin{align*}
P(C_{a} | M = B) &= P(C_{c}|M=B) \\
\frac{(1-\alpha) \pi_{a}}{(1-\alpha) \pi_{a} + \pi_{c}} &= \frac{1 \cdot \pi_{c}}{(1-\alpha) \pi_{a} + \pi_{c}} \\
\pi_{c} &= (1-\alpha)\pi_{a}
\end{align*}

I perform simulation to confirm analytic solutions, and solve for the priors that ensure switching is no longer advanageous.

In [2]:

"""
Monty Hall simulation with arbitrary priors and Monty's bias (alpha).

Rules:
- Player initially chooses door A.
- Priors: pi_A, pi_B, pi_C (sum to 1).
- If car is at A, Monty opens C with prob alpha (else opens B).
- If car is at B, Monty must open C.
- If car is at C, Monty must open B.

Estimate the posteriors:
  P(C_A | M=C) and P(C_B | M=C),
  P(C_A | M=B) and P(C_C | M=B),
and verify the indifference conditions:
  pi_B = alpha * pi_A (for M=C),
  pi_C = (1 - alpha) * pi_A (for M=B).
"""

#Standard imports
from __future__ import annotations
import numpy as np
import pandas as pd

#Utility function for safe division
def safe_div(num, den):
    if den == 0:
        return float("nan")
    else:
        return float(num) / float(den)


#Function to perform simulation
def simulate(n: int, pi_A: float, pi_B: float, pi_C: float, alpha: float, seed: int | None = None):
    """
    Perform a Monte Carlo simulation of the Monty Hall problem.

    Parameters
    ----------
    n : int, Number of trials.
    pi_A, pi_B, pi_C : float, Prior probabilities that the car is behind doors A, B, C respectively (must sum to 1).
    alpha : float, Monty's bias: probability Monty opens door C when the car is at A (your initial door).
                   Thus Monty opens B with probability (1 - alpha) when the car is at A.
    seed : Optional[int], Seed for reproducibility.

    Returns
    -------
    sim_results: dict, a summary dictionary including posteriors and switch win rates conditioned on Monty's action.
    """

    #Check for valid inputs. Priors must be non-negative.
    for pi in [pi_A, pi_B, pi_C]:
        if not (0.0 <= pi <= 1.0):
            raise ValueError("Priors must be in [0, 1]")
    #Priors must sum to 1.
    if not np.isclose( pi_A + pi_B + pi_C, 1.0):
        raise ValueError("Priors must sum to 1.")
    #Bias must be valid probability
    if not (0.0 <= alpha <= 1.0):
        raise ValueError("alpha must be in [0, 1].")

    #Set random seed for reproducability
    rng = np.random.default_rng(seed)

    #Encode doors A, B, C as ints 0, 1, 2, respectively
    doors = [0,1,2]

    #Set priors
    probs = np.array([pi_A, pi_B, pi_C], dtype=float)

    #Choose door concealing car per specified prior.
    car = rng.choice(doors, size=n, p=probs)

    #Monty's decision: monty==1 (open B) or monty==2 (open C).
    monty = np.empty(n, dtype=np.int8)

    """
    3 cases:
    - Case car==A (0): Monty opens door C with prob. alpha and B with prob. 1-alpha
    - Case car==B (1): Monty must open door C (2)
    - Case car==C (2): Monty must open door B (1)
    """

    #Case car==A (0):  open C (2) w.p. alpha, B (1) o.t.w
    maskA = (car == 0)
    numA = maskA.sum()
    if numA:
        u = rng.random(numA)
        monty[maskA] = np.where(u < alpha, 2, 1)

    #Case car==B (1): open door C (2)
    maskB = (car == 1)
    monty[maskB] = 2

    #Case car==C (2): open door B (1)
    maskC = (car == 2)
    monty[maskC] = 1

    #Condition on Monty opened C (2) or B (1)
    MC = (monty == 2)
    MB = (monty == 1)

    #Compute posteriors after M=C
    p_CA_given_MC = safe_div(np.sum((car == 0) & MC), np.sum(MC))
    p_CB_given_MC = safe_div(np.sum((car == 1) & MC), np.sum(MC))


    #Compute posteriors after M=B
    p_CA_given_MB = safe_div(np.sum((car == 0) & MB), np.sum(MB))
    p_CC_given_MB = safe_div(np.sum((car == 2) & MB), np.sum(MB))

    #Structure and return results
    sim_results = {"pi_A": pi_A, "pi_B": pi_B, "pi_C": pi_C, "alpha": alpha, #Priors and bias
                   "P(C_A | M=C)": p_CA_given_MC, "P(C_B | M=C)": p_CB_given_MC, #Posterior
                   "P(C_A | M=B)": p_CA_given_MB, "P(C_C | M=B)": p_CC_given_MB, #Posterior
                   "IC(M=C): pi_B = alpha*pi_A": alpha * pi_A, #Indifference condition when M = C
                   "IC(M=B): pi_C = (1-alpha)*pi_A": (1 - alpha) * pi_A, #Indifference condition when M = B
                   "Pr(M=C)": float(np.mean(MC)), #Simulation 'reveal C' rate
                   "Pr(M=B)": float(np.mean(MB)), #Simulation 'reveal B' rate
                   }
    return sim_results

#Main function call
if __name__ == "__main__":

    """
    Test the simulation function with 3 different cases:
    1. Classic symmetric case: expect switch ~= 2/3
    2. Indifference after M=C: set pi_B = alpha * pi_A
    3. Indifference after M=B: set pi_C = (1 - alpha) * pi_A
    4. Indifference after M=B or M=C: set pi_B = alpha * pi_A, pi_C = (1 - alpha) * pi_A => pi_A = 0.5
    Set simulation parameters: n=1_000_000, pi_A=pi_B=pi_C=1/3, alpha=0.5, seed=1
    """
    #Case 1 (classic)
    res_1 = simulate(n=1_000_000, pi_A=1/3, pi_B=1/3, pi_C=1/3, alpha=0.5, seed=1)

    #Case 2 (indifference after M=C)
    res_2 = simulate(n=1_000_000, pi_A=0.4, pi_B=0.12, pi_C=0.48, alpha=0.3, seed=1)

    #Case 3 (indifference after M=B)
    res_3 = simulate(n=1_000_000, pi_A=1/3, pi_B=1/2, pi_C=1/6, alpha=0.5, seed=1)

    #Case 4 (indifference regardless of M=B or M=C)
    res_4 = simulate(n=1_000_000, pi_A=0.5, pi_B=0.125, pi_C=0.375, alpha=0.25, seed=1)

    #Check results
    display(pd.DataFrame([res_1, res_2, res_3, res_4], index=["Classic","Indifference M=C","Indifference M=B", "Indifference (M=B or M=C)"]).T)


,Classic,Indifference M=C,Indifference M=B,Indifference (M=B or M=C)
pi_A,0.333333,0.400000,0.333333,0.500000
pi_B,0.333333,0.120000,0.500000,0.125000
pi_C,0.333333,0.480000,0.166667,0.375000
alpha,0.500000,0.300000,0.500000,0.250000
P(C_A | M=C),0.333049,0.499636,0.249830,0.500882
P(C_B | M=C),0.666951,0.500364,0.750170,0.499118
P(C_A | M=B),0.333607,0.368971,0.499976,0.500201
P(C_C | M=B),0.666393,0.631029,0.500024,0.499799
IC(M=C): pi_B = alpha*pi_A,0.166667,0.120000,0.166667,0.125000
IC(M=B): pi_C = (1-alpha)*pi_A,0.166667,0.280000,0.166667,0.375000


To demonstrate the indifference conditions, I solve for valid priors and bias. The case M=C:

\begin{align*}
\pi_{a} + \pi_{b} + \pi_{c} &= 1\\
\pi_{b} &= \alpha \pi_{a} \\
(1+ \alpha)\pi_{a} + \pi_{c} &= 1
\end{align*}

has, for example, $\pi_{a} = 0.4, \pi_{b} = 0.12, \pi_{c} = 0.48, \alpha=0.3$ as a solution. The case M=B:

\begin{align*}
\pi_{a} + \pi_{b} + \pi_{c} &= 1\\
\pi_{c} &= (1-\alpha) \pi_{a} \\
\pi_{b} + (2-\alpha) \pi_{a} &= 1
\end{align*}

has, for example, $\pi_{a} = 1/3, \pi_{b} = 1/2, \pi_{c} = 1/6, \alpha=0.5$ as a solution.

It's possible for you to be indifferent between switching, regardless of which door (B or C) Monty reveals. Substitute both indifference conditions:

\begin{align*}
\pi_{a} + \pi_{b} + \pi_{c} &= 1\\
\pi_{c} &= (1-\alpha) \pi_{a} \\
\pi_{b} &= \alpha \pi_{a} \\
\pi_{a} + \alpha \pi_{a} + (1-\alpha) \pi_{a} &= 1\\
\pi_{a} = 0.5\\
\end{align*}

Regardless of Monty's switching bias $\alpha$, for priors $\pi_{a} = 0.5, \pi_{b} = 0.5\alpha, \pi_{c}=0.5(1-\alpha)$, all posteriors are the same. A "strategic" Monty can ensure players don't systematically win over many iterations of the game by using the above priors when choosing which door conceals the goat.

Simulation results under the classic Monty Hall conditions are as expected. Switching from door A increases the win rate by $1/3$, regardless of which door Monty opens. For the indifference conditions, posteriors are equal as expected. For example, using simulated parameters chosen to induce indifference when Monty reveals B (M=B) produces posteriors $P(C_{a}|M=B) = P(C_{c}|M=B)$.